# GIF Profila Brzina i Pritiska

```powershell
SDL_VIDEODRIVER=x11 python src/pygame_viewer_venturi.py --geometry venturi --frames 600 --export-profile-csv exports/venturi_profile.csv --export-every 2 --viscosity 0 --damping 100
SDL_VIDEODRIVER=x11 python src/pygame_viewer_venturi.py --geometry flat --frames 600 --export-profile-csv exports/flat_profile.csv --export-every 2 --viscosity 0.1 --damping 100
```

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
import pandas as pd

plt.style.use("seaborn-v0_8-whitegrid")

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
PROFILE_CSV = PROJECT_ROOT / "exports" / "venturi_profile.csv"
OUTPUT_DIR = PROJECT_ROOT / "exports"
OUTPUT_DIR.mkdir(exist_ok=True)

profile = pd.read_csv(PROFILE_CSV)
profile = profile[profile["frame"] > 0].copy()
profile.head()

In [ ]:
frames = sorted(profile["frame"].unique())
geometry = profile["geometry"].iloc[0] if "geometry" in profile else "simulation"
x_min, x_max = profile["x"].min(), profile["x"].max()
velocity_limits = (profile["velocity_x"].min(), profile["velocity_x"].max())
speed_limits = (profile["speed"].min(), profile["speed"].max())
pressure_limits = (profile["pressure"].min(), profile["pressure"].max())

def padded(limits, pad=0.08):
    lo, hi = limits
    if lo == hi:
        return lo - 1, hi + 1
    extra = (hi - lo) * pad
    return lo - extra, hi + extra

In [ ]:
def save_profile_gif(column, ylabel, output_name, limits, fps=20):
    fig, ax = plt.subplots(figsize=(9, 4.8))
    (line,) = ax.plot([], [], linewidth=2)
    title = ax.set_title("")
    ax.set_xlim(x_min, x_max)
    ax.set_ylim(*padded(limits))
    ax.set_xlabel("x cell")
    ax.set_ylabel(ylabel)

    def update(frame):
        row = profile[profile["frame"] == frame]
        line.set_data(row["x"], row[column])
        time_value = row["time"].iloc[0] if "time" in row else frame
        title.set_text(f"{geometry}: {ylabel} vs x | frame {frame} | t={time_value:.3f}")
        return line, title

    animation = FuncAnimation(fig, update, frames=frames, interval=1000 / fps, blit=False)
    output_path = OUTPUT_DIR / output_name
    animation.save(output_path, writer=PillowWriter(fps=fps))
    plt.close(fig)
    return output_path

In [ ]:
velocity_gif = save_profile_gif(
    "velocity_x",
    "centerline velocity_x",
    f"{PROFILE_CSV.stem}_velocity_x.gif",
    velocity_limits,
)
pressure_gif = save_profile_gif(
    "pressure",
    "mean pressure",
    f"{PROFILE_CSV.stem}_pressure.gif",
    pressure_limits,
)
speed_gif = save_profile_gif(
    "speed",
    "mean speed",
    f"{PROFILE_CSV.stem}_speed.gif",
    speed_limits,
)

velocity_gif, pressure_gif, speed_gif